# Notebook 03 — Query Routing: gemma3 Classifier & Cypher Generator
**Layer:** Retrieval routing  
**Model:** `gemma3` via local ollama — no API key, no rate limits, no internet  
**Inputs:** User query · Neo4j (from NB01)  
**Outputs:** `intent_type` · `slots` JSON · Cypher result

## Setup — run once
```bash
ollama pull gemma3
```

## 3.1 Imports, ollama client, Neo4j driver

In [1]:
import json
import re
from neo4j import GraphDatabase
import ollama

# Local ollama — no API key, no rate limits
OLLAMA_MODEL = "gemma3"

# Verify ollama is running and model is available
models = [m.model for m in ollama.list().models]
if OLLAMA_MODEL not in models and not any(OLLAMA_MODEL in m for m in models):
    print(f"WARNING: {OLLAMA_MODEL} not found. Run: ollama pull {OLLAMA_MODEL}")
    print(f"Available models: {models}")
else:
    print(f"ollama model ready: {OLLAMA_MODEL}")

NEO4J_URI  = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASS = "pass@Word123"   # <-- update

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS))
print(f"Neo4j driver ready — {NEO4J_URI}")

ollama model ready: gemma3
Neo4j driver ready — bolt://localhost:7687


## 3.2 Ollama call wrapper and valid values

In [2]:
def call_ollama(prompt: str, system: str = "", fmt: str = "") -> str:
    """
    Call gemma3 via local ollama. No API key, no rate limits, no internet.
    - prompt : user message
    - system : optional system instruction
    - fmt    : pass "json" to request JSON-only output
    Returns the response text string.
    """
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    kwargs = {"model": OLLAMA_MODEL, "messages": messages}
    if fmt == "json":
        kwargs["format"] = "json"

    response = ollama.chat(**kwargs)
    return response.message.content.strip()


VALID_TOWNS = [
    "ANG MO KIO","BEDOK","BISHAN","BUKIT BATOK","BUKIT MERAH","BUKIT PANJANG",
    "BUKIT TIMAH","CENTRAL AREA","CHOA CHU KANG","CLEMENTI","GEYLANG","HOUGANG",
    "JURONG EAST","JURONG WEST","KALLANG/WHAMPOA","MARINE PARADE","PASIR RIS",
    "PUNGGOL","QUEENSTOWN","SEMBAWANG","SENGKANG","SERANGOON","TAMPINES",
    "TOA PAYOH","WOODLANDS","YISHUN",
]
VALID_FLAT_TYPES = [
    "1 ROOM","2 ROOM","3 ROOM","4 ROOM","5 ROOM","EXECUTIVE","MULTI-GENERATION"
]
print("call_ollama() and valid values defined.")

call_ollama() and valid values defined.


## 3.3 Classifier prompt and classify_query()

In [3]:
CLASSIFIER_SYSTEM = """You are a query classifier for an HDB flat database system.

Classify the user query into EXACTLY ONE of these intent types:
- PRICE_ESTIMATION: user wants a price estimate or valuation
- NEIGHBOURHOOD: user wants nearby amenities (MRT, malls, food courts, highways)
- SCHOOL_CATCHMENT: user wants school proximity or quality
- INVESTMENT_TEMPORAL: user wants historical price trends or town-level appreciation
- LEASE_ADVISORY: user wants guidance related to lease remaining years

Extract these slots if mentioned (null if not mentioned):
- town: must be one of the 26 valid HDB towns (uppercase)
- flat_type: one of [1 ROOM, 2 ROOM, 3 ROOM, 4 ROOM, 5 ROOM, EXECUTIVE, MULTI-GENERATION]
- room_count: integer 1-6
- budget_sgd_min: integer
- budget_sgd_max: integer
- year_min: integer 2015-2026
- year_max: integer 2015-2026
- lease_years_max: integer

You MUST respond with ONLY a valid JSON object. No explanation, no markdown.
Format: {"intent": "...", "slots": {"town": null, "flat_type": null, "room_count": null,
"budget_sgd_min": null, "budget_sgd_max": null, "year_min": null,
"year_max": null, "lease_years_max": null}}
"""


def classify_query(user_query: str) -> dict:
    prompt = "Classify this query and extract slots:\n\n" + user_query
    raw    = call_ollama(prompt, system=CLASSIFIER_SYSTEM, fmt="json")
    raw    = re.sub(r"```json|```", "", raw).strip()
    return json.loads(raw)


# Test all 5 intent types
test_queries = [
    "How much is a 4-room flat in Bishan worth?",
    "What amenities are near Ang Mo Kio flats?",
    "Which areas have the best primary schools under $700K?",
    "Which town had the best price growth from 2020 to 2025?",
    "Should I buy a flat with only 55 years of lease left?",
]
for q in test_queries:
    result = classify_query(q)
    print(f"Q: {q[:55]}")
    print(f"   Intent: {result['intent']} | Slots: {result['slots']}\n")

Q: How much is a 4-room flat in Bishan worth?
   Intent: PRICE_ESTIMATION | Slots: {'town': 'BISHAN', 'flat_type': '4 ROOM', 'room_count': 4}

Q: What amenities are near Ang Mo Kio flats?
   Intent: NEIGHBOURHOOD | Slots: {'town': 'ANG MO KIO', 'flat_type': None, 'room_count': None, 'budget_sgd_min': None, 'budget_sgd_max': None, 'year_min': None, 'year_max': None, 'lease_years_max': None}

Q: Which areas have the best primary schools under $700K?
   Intent: SCHOOL_CATCHMENT | Slots: {'town': None, 'flat_type': None, 'room_count': None, 'budget_sgd_min': 700000, 'budget_sgd_max': 700000, 'year_min': None, 'year_max': None, 'lease_years_max': None}

Q: Which town had the best price growth from 2020 to 2025?
   Intent: INVESTMENT_TEMPORAL | Slots: {'town': None, 'flat_type': None, 'room_count': None, 'budget_sgd_min': None, 'budget_sgd_max': None, 'year_min': 2020, 'year_max': 2025, 'lease_years_max': None}

Q: Should I buy a flat with only 55 years of lease left?
   Intent: LEASE_ADVISO

## 3.4 Cypher template library

In [4]:
CYPHER_TEMPLATES = {

    "PRICE_ESTIMATION": """
MATCH (f:Flat)-[:IN_TOWN]->(t:Town)
WHERE ($town IS NULL OR t.name = $town)
  AND ($flat_type IS NULL OR f.flat_type = $flat_type)
  AND ($room_count IS NULL OR f.room_count = $room_count)
  AND ($budget_sgd_min IS NULL OR f.resale_price >= $budget_sgd_min)
  AND ($budget_sgd_max IS NULL OR f.resale_price <= $budget_sgd_max)
RETURN percentileCont(f.resale_price,0.5) AS median_price,
       avg(f.resale_price) AS avg_price, stDev(f.resale_price) AS std_price,
       count(f) AS tx_count, avg(f.lease_remaining_years) AS avg_lease,
       avg(f.floor_area_sqm) AS avg_area,
       min(f.transaction_year) AS year_min, max(f.transaction_year) AS year_max""",

    "NEIGHBOURHOOD": """
MATCH (f:Flat)-[:IN_TOWN]->(t:Town)
WHERE ($town IS NULL OR t.name = $town)
RETURN t.name AS town, avg(f.dist_to_mrt_m) AS avg_mrt_dist_m,
       avg(f.dist_to_foodcourt_m) AS avg_foodcourt_dist_m,
       avg(f.dist_to_nearest_mall_m) AS avg_mall_dist_m,
       avg(f.mall_count_3km) AS avg_mall_count_3km,
       avg(f.mall_weighted_access_3km) AS avg_mall_access_score,
       avg(f.dist_to_highway_m) AS avg_highway_dist_m,
       count(f) AS flat_count""",

    "SCHOOL_CATCHMENT": """
MATCH (f:Flat)-[:IN_TOWN]->(t:Town)
WHERE ($town IS NULL OR t.name = $town)
  AND ($budget_sgd_max IS NULL OR f.resale_price <= $budget_sgd_max)
  AND ($flat_type IS NULL OR f.flat_type = $flat_type)
RETURN t.name AS town,
       avg(f.primary_school_quality_1km_weighted) AS avg_school_quality,
       avg(f.primary_school_top_quality_1km) AS avg_top_school_quality,
       avg(f.school_count_1km) AS avg_schools_in_1km,
       avg(f.dist_to_nearest_school_m) AS avg_school_dist_m,
       avg(f.resale_price) AS avg_price, count(f) AS flat_count
ORDER BY avg_school_quality DESC""",

    "INVESTMENT_TEMPORAL": """
MATCH (f:Flat)-[:IN_TOWN]->(t:Town)
WHERE ($town IS NULL OR t.name = $town)
  AND ($year_min IS NULL OR f.transaction_year >= $year_min)
  AND ($year_max IS NULL OR f.transaction_year <= $year_max)
RETURN t.name AS town, f.transaction_year AS year,
       avg(f.resale_price) AS avg_price,
       percentileCont(f.resale_price,0.5) AS median_price,
       count(f) AS tx_count
ORDER BY t.name, f.transaction_year""",

    "LEASE_ADVISORY": """
MATCH (f:Flat)-[:IN_TOWN]->(t:Town)
WHERE ($town IS NULL OR t.name = $town)
  AND ($lease_years_max IS NULL OR f.lease_remaining_years <= $lease_years_max)
RETURN CASE
         WHEN f.lease_remaining_years < 50 THEN '<50 years'
         WHEN f.lease_remaining_years < 70 THEN '50-69 years'
         ELSE '70+ years'
       END AS lease_band,
       avg(f.resale_price) AS avg_price,
       percentileCont(f.resale_price,0.5) AS median_price,
       count(f) AS tx_count
ORDER BY lease_band"""
}
print("Cypher templates loaded:", list(CYPHER_TEMPLATES.keys()))

Cypher templates loaded: ['PRICE_ESTIMATION', 'NEIGHBOURHOOD', 'SCHOOL_CATCHMENT', 'INVESTMENT_TEMPORAL', 'LEASE_ADVISORY']


## 3.5 Slot mapper and end-to-end routing test

In [7]:
def slots_to_params(slots: dict) -> dict:
    town      = slots.get("town")
    flat_type = slots.get("flat_type")
    if town and town.upper() not in VALID_TOWNS:          town = None
    if flat_type and flat_type.upper() not in VALID_FLAT_TYPES: flat_type = None
    return {
        "town":            town.upper() if town else None,
        "flat_type":       flat_type.upper() if flat_type else None,
        "room_count":      slots.get("room_count"),
        "budget_sgd_min":  slots.get("budget_sgd_min"),
        "budget_sgd_max":  slots.get("budget_sgd_max"),
        "year_min":        slots.get("year_min"),
        "year_max":        slots.get("year_max"),
        "lease_years_max": slots.get("lease_years_max"),
    }


def route_and_query(user_query: str) -> dict:
    classified = classify_query(user_query)
    intent     = classified["intent"]
    params     = slots_to_params(classified["slots"])
    cypher     = CYPHER_TEMPLATES[intent]
    with driver.session() as session:
        records = [dict(r) for r in session.run(cypher, **params)]
    return {"intent": intent, "slots": classified["slots"],
            "params": params, "records": records}


for q in test_queries:
    out = route_and_query(q)
    print(f"Intent: {out['intent']}")
    print(f"  Params : {out['params']}")
    print(f"  Records: {len(out['records'])}")
    if out["records"]:
        print(f"  First  : {out['records'][0]}")
    print()

driver.close()
print("Notebook 03 complete.")

Intent: PRICE_ESTIMATION
  Params : {'town': 'BISHAN', 'flat_type': '4 ROOM', 'room_count': 4, 'budget_sgd_min': None, 'budget_sgd_max': None, 'year_min': None, 'year_max': None, 'lease_years_max': None}
  Records: 1
  First  : {'median_price': None, 'avg_price': None, 'std_price': 0.0, 'tx_count': 0, 'avg_lease': None, 'avg_area': None, 'year_min': None, 'year_max': None}

Intent: NEIGHBOURHOOD
  Params : {'town': 'ANG MO KIO', 'flat_type': None, 'room_count': None, 'budget_sgd_min': None, 'budget_sgd_max': None, 'year_min': None, 'year_max': None, 'lease_years_max': None}
  Records: 0

Intent: SCHOOL_CATCHMENT
  Params : {'town': None, 'flat_type': None, 'room_count': None, 'budget_sgd_min': 700000, 'budget_sgd_max': 700000, 'year_min': None, 'year_max': None, 'lease_years_max': None}
  Records: 0

Intent: INVESTMENT_TEMPORAL
  Params : {'town': None, 'flat_type': None, 'room_count': None, 'budget_sgd_min': None, 'budget_sgd_max': None, 'year_min': 2020, 'year_max': 2025, 'lease_year